In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
 █▓▒░ LEVIATÁN DEFINITIVO TSM: ARQUITECTURA Z/6Z UNIFICADA ░▒▓█
=============================================================================
 Fusión Estratégica:
 - Logística TSM (DeepSeek): Calibración de Airy, Fricción Quiral.
 - Motor Termodinámico (Original): OpenMP L1, Espada Z/6Z completa y ECM con
   Acumulador Z (Parada Prematura real).
 - Orquestación Asíncrona: ProcessPoolExecutor para victoria instantánea.
=============================================================================
"""

import os
import subprocess
import ctypes
import time
import math
import random
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed

# =========================================================================
# 0. INSTALAR DEPENDENCIAS
# =========================================================================
print("📦 Verificando blindaje y dependencias...")
try:
    import gmpy2
except ImportError:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "libgmp-dev", "g++", "-qq"], check=False)
    subprocess.run(["pip", "install", "gmpy2", "-q"], check=False)
    import gmpy2

# =========================================================================
# 1. FORJA DEL NÚCLEO C++ (MOTOR TERMODINÁMICO CORREGIDO)
# =========================================================================
cpp_code = """
#include <iostream>
#include <vector>
#include <string>
#include <cmath>
#include <cstring>
#include <gmp.h>
#include <omp.h>

using namespace std;

// ... [AQUÍ VA EL RESTO DE TU CÓDIGO C++ CON EL p_count CORREGIDO] ...

static char out_buf_criba[10000];
static char out_buf_fase0[2048];
static char out_buf_ecm[2048];

extern "C" {
    // --- FASE 1: VANGUARDIA Z/6Z (CRIBA OPENMP DE ALTA DENSIDAD) ---
    const char* criba_divide_cpp(const char* n_str, uint64_t limite) {
        mpz_t N; mpz_init_set_str(N, n_str, 10);
        string result = "";
        while (mpz_divisible_ui_p(N, 2)) { result += "2,"; mpz_divexact_ui(N, N, 2); }
        while (mpz_divisible_ui_p(N, 3)) { result += "3,"; mpz_divexact_ui(N, N, 3); }

        if (mpz_cmp_ui(N, 1) > 0 && limite >= 5) {
            uint64_t k_max = limite / 6;
            uint64_t num_words = (k_max / 64) + 1;
            vector<uint64_t> C1(num_words, ~0ULL);
            vector<uint64_t> C5(num_words, ~0ULL);
            C1[0] &= ~1ULL;
            uint64_t raiz = sqrt(limite);

            #pragma omp parallel for schedule(dynamic)
            for (uint64_t k = 0; k <= raiz/6 + 1; k++) {
                uint64_t p1 = 6*k + 1;
                if (p1 > 1 && p1 <= raiz) {
                    for (uint64_t i = (p1*p1-1)/6; i <= k_max; i+=p1) C1[i/64] &= ~(1ULL << (i%64));
                    for (uint64_t i = (p1*(p1+4)-5)/6; i <= k_max; i+=p1) C5[i/64] &= ~(1ULL << (i%64));
                }
                uint64_t p5 = 6*k + 5;
                if (p5 <= raiz) {
                    for (uint64_t i = (p5*p5-1)/6; i <= k_max; i+=p5) C1[i/64] &= ~(1ULL << (i%64));
                    for (uint64_t i = (p5*(p5+2)-5)/6; i <= k_max; i+=p5) C5[i/64] &= ~(1ULL << (i%64));
                }
            }
            for (uint64_t k = 0; k <= k_max && mpz_cmp_ui(N, 1) > 0; k++) {
                if (C1[k/64] & (1ULL << (k%64))) {
                    uint64_t p = 6*k + 1;
                    while (mpz_divisible_ui_p(N, p)) { result += to_string(p) + ","; mpz_divexact_ui(N, N, p); }
                }
                if (C5[k/64] & (1ULL << (k%64))) {
                    uint64_t p = 6*k + 5;
                    while (mpz_divisible_ui_p(N, p)) { result += to_string(p) + ","; mpz_divexact_ui(N, N, p); }
                }
            }
        }
        char res_buf[2048]; mpz_get_str(res_buf, 10, N); result += string(res_buf);
        strncpy(out_buf_criba, result.c_str(), sizeof(out_buf_criba)-1);
        mpz_clear(N); return out_buf_criba;
    }

    // --- FASE 2: FRANCOTIRADOR ESPADA (FERMAT Z/6Z CANALES 1 Y 5) ---
    const char* espada_cpp(const char* n_str, int canal, uint64_t iters) {
        mpz_t n, x, y2, y, f, m, m3, x2;
        mpz_inits(n, x, y2, y, f, m, m3, x2, NULL);
        mpz_set_str(n, n_str, 10);
        mpz_sqrt(x, n);
        bool encontrado = false;

        if (canal == 5) {
            uint64_t r = mpz_fdiv_ui(x, 3);
            if (r != 0) mpz_add_ui(x, x, 3 - r);
            for (uint64_t i = 0; i < iters; i++) {
                mpz_mul(y2, x, x); mpz_sub(y2, y2, n);
                if (mpz_sgn(y2) >= 0 && mpz_perfect_square_p(y2)) {
                    mpz_sqrt(y, y2); mpz_sub(f, x, y);
                    encontrado = true; break;
                }
                mpz_add_ui(x, x, 3);
            }
        } else if (canal == 1) {
            mpz_set_ui(m, 1);
            for (uint64_t i = 0; i < iters; i++) {
                mpz_mul_ui(m3, m, 3); mpz_mul(x2, m3, m3); mpz_add(x2, x2, n);
                if (mpz_perfect_square_p(x2)) {
                    mpz_sqrt(x, x2); mpz_sub(f, x, m3);
                    encontrado = true; break;
                }
                mpz_add_ui(m, m, 1);
            }
        }

        if (encontrado) mpz_get_str(out_buf_fase0, 10, f);
        mpz_clears(n, x, y2, y, f, m, m3, x2, NULL);
        return encontrado ? out_buf_fase0 : nullptr;
    }

    // --- FASE 3: ARTILLERÍA ECM (ACUMULADOR Z Y PARADA PREMATURA) ---
    void montgomery_mul(mpz_t X, mpz_t Z, mpz_t k, mpz_t a24, mpz_t n) {
        mpz_t x0, z0, x1, z1, u, v, w, t1, t2, t3;
        mpz_inits(x0, z0, x1, z1, u, v, w, t1, t2, t3, NULL);
        mpz_set(x0, X); mpz_set(z0, Z);
        mpz_add(u, x0, z0); mpz_mul(u, u, u); mpz_mod(u, u, n);
        mpz_sub(v, x0, z0); mpz_mul(v, v, v); mpz_mod(v, v, n);
        mpz_sub(w, u, v); mpz_mod(w, w, n);
        mpz_mul(x1, u, v); mpz_mod(x1, x1, n);
        mpz_mul(t1, a24, w); mpz_add(t1, t1, v); mpz_mod(t1, t1, n);
        mpz_mul(z1, w, t1); mpz_mod(z1, z1, n);
        size_t bits = mpz_sizeinbase(k, 2);
        for (long i = bits - 2; i >= 0; i--) {
            int bit = mpz_tstbit(k, i);
            if (bit == 1) {
                mpz_sub(t1, x0, z0); mpz_add(t2, x1, z1); mpz_mul(t1, t1, t2); mpz_mod(t1, t1, n);
                mpz_add(t2, x0, z0); mpz_sub(t3, x1, z1); mpz_mul(t2, t2, t3); mpz_mod(t2, t2, n);
                mpz_add(u, t1, t2); mpz_mul(u, u, u); mpz_mod(u, u, n);
                mpz_sub(v, t1, t2); mpz_mul(v, v, v); mpz_mod(v, v, n);
                mpz_mul(x0, Z, u); mpz_mod(x0, x0, n);
                mpz_mul(z0, X, v); mpz_mod(z0, z0, n);
                mpz_add(u, x1, z1); mpz_mul(u, u, u); mpz_mod(u, u, n);
                mpz_sub(v, x1, z1); mpz_mul(v, v, v); mpz_mod(v, v, n);
                mpz_sub(w, u, v); mpz_mod(w, w, n);
                mpz_mul(x1, u, v); mpz_mod(x1, x1, n);
                mpz_mul(t1, a24, w); mpz_add(t1, t1, v); mpz_mod(t1, t1, n);
                mpz_mul(z1, w, t1); mpz_mod(z1, z1, n);
            } else {
                mpz_sub(t1, x1, z1); mpz_add(t2, x0, z0); mpz_mul(t1, t1, t2); mpz_mod(t1, t1, n);
                mpz_add(t2, x1, z1); mpz_sub(t3, x0, z0); mpz_mul(t2, t2, t3); mpz_mod(t2, t2, n);
                mpz_add(u, t1, t2); mpz_mul(u, u, u); mpz_mod(u, u, n);
                mpz_sub(v, t1, t2); mpz_mul(v, v, v); mpz_mod(v, v, n);
                mpz_mul(x1, Z, u); mpz_mod(x1, x1, n);
                mpz_mul(z1, X, v); mpz_mod(z1, z1, n);
                mpz_add(u, x0, z0); mpz_mul(u, u, u); mpz_mod(u, u, n);
                mpz_sub(v, x0, z0); mpz_mul(v, v, v); mpz_mod(v, v, n);
                mpz_sub(w, u, v); mpz_mod(w, w, n);
                mpz_mul(x0, u, v); mpz_mod(x0, x0, n);
                mpz_mul(t1, a24, w); mpz_add(t1, t1, v); mpz_mod(t1, t1, n);
                mpz_mul(z0, w, t1); mpz_mod(z0, z0, n);
            }
        }
        mpz_set(X, x0); mpz_set(Z, z0);
        mpz_clears(x0, z0, x1, z1, u, v, w, t1, t2, t3, NULL);
    }

    const char* ecm_final_cpp(const char* n_str, double sigma_seed, int b1, int curvas) {
        mpz_t n, factor, u, v, X, Z, a, b, a24, inv, sigma, t1, t2, k_pow, acum_Z;
        mpz_inits(n, factor, u, v, X, Z, a, b, a24, inv, sigma, t1, t2, k_pow, acum_Z, NULL);
        mpz_set_str(n, n_str, 10);
        bool encontrado = false;

        vector<bool> is_p(b1 + 1, true);
        for (int p = 2; p * p <= b1; p++) if (is_p[p]) for (int i = p * p; i <= b1; i += p) is_p[i] = false;
        vector<uint32_t> primes;
        if (b1 >= 2) primes.push_back(2); if (b1 >= 3) primes.push_back(3);
        for (int k = 1; 6 * k - 1 <= b1; k++) {
            if (is_p[6 * k - 1]) primes.push_back(6 * k - 1);
            if (6 * k + 1 <= b1 && is_p[6 * k + 1]) primes.push_back(6 * k + 1);
        }

        for (int i = 0; i < curvas; i++) {
            mpz_set_d(sigma, sigma_seed + i);
            mpz_mul(u, sigma, sigma); mpz_sub_ui(u, u, 5); mpz_mul_ui(v, sigma, 4);
            mpz_pow_ui(X, u, 3); mpz_pow_ui(Z, v, 3);
            mpz_sub(t1, v, u); mpz_pow_ui(t1, t1, 3); mpz_mul_ui(t2, u, 3); mpz_add(t2, t2, v); mpz_mul(a, t1, t2); mpz_mod(a, a, n);
            mpz_pow_ui(t1, u, 3); mpz_mul_ui(t1, t1, 4); mpz_mul(b, t1, v); mpz_mod(b, b, n);
            mpz_mul_ui(inv, b, 4);
            if (mpz_invert(inv, inv, n) == 0) {
                mpz_gcd(factor, inv, n);
                if (mpz_cmp_ui(factor, 1) > 0 && mpz_cmp(factor, n) < 0) { encontrado = true; break; }
            }
            mpz_mul(a24, a, inv); mpz_mod(a24, a24, n);
            mpz_set_ui(acum_Z, 1);

            // LA CLAVE TÁCTICA ORIGINAL: Acumulador Z y parada cada 256 pasos
            // CORRECCIÓN TÁCTICA: Contador real para la parada prematura
            int p_count = 0;
            for (uint32_t p : primes) {
                uint32_t pwr = log(b1) / log(p); mpz_ui_pow_ui(k_pow, p, pwr);
                montgomery_mul(X, Z, k_pow, a24, n);
                mpz_mul(acum_Z, acum_Z, Z); mpz_mod(acum_Z, acum_Z, n);

                p_count++;
                if (p_count % 256 == 0) {
                    mpz_gcd(factor, acum_Z, n);
                    if (mpz_cmp_ui(factor, 1) > 0 && mpz_cmp(factor, n) < 0) { encontrado = true; break; }
                    mpz_set_ui(acum_Z, 1);
                }
            }
            if (!encontrado) { mpz_gcd(factor, acum_Z, n); if (mpz_cmp_ui(factor, 1) > 0 && mpz_cmp(factor, n) < 0) encontrado = true; }
            if (encontrado) break;
        }
        if (encontrado) mpz_get_str(out_buf_ecm, 10, factor);
        mpz_clears(n, factor, u, v, X, Z, a, b, a24, inv, sigma, t1, t2, k_pow, acum_Z, NULL);
        return encontrado ? out_buf_ecm : nullptr;
    }
}
"""

with open("leviatan_final_core.cpp", "w") as f: f.write(cpp_code)
print("[🔨] Forjando Núcleo C++ Optimizado...")
result = subprocess.run(["g++", "-O3", "-fopenmp", "-shared", "-fPIC", "-o", "leviatan_final_core.so", "leviatan_final_core.cpp", "-lgmp"], capture_output=True, text=True)
if result.returncode != 0: raise RuntimeError(f"Error C++: {result.stderr}")
print("[✅] Núcleo operativo y blindado.")

L_LIB = ctypes.CDLL(os.path.abspath('./leviatan_final_core.so'))
L_LIB.criba_divide_cpp.argtypes = [ctypes.c_char_p, ctypes.c_uint64]; L_LIB.criba_divide_cpp.restype = ctypes.c_char_p
L_LIB.espada_cpp.argtypes = [ctypes.c_char_p, ctypes.c_int, ctypes.c_uint64]; L_LIB.espada_cpp.restype = ctypes.c_char_p
L_LIB.ecm_final_cpp.argtypes = [ctypes.c_char_p, ctypes.c_double, ctypes.c_int, ctypes.c_int]; L_LIB.ecm_final_cpp.restype = ctypes.c_char_p

# =========================================================================
# 2. CONSTANTES MATEMÁTICAS TSM (DEEPSEEK LOGISTICS)
# =========================================================================
R_FUND = math.log(2) / (6 * math.log(3))
KAPPA_INFO = math.log(2) / (4 * math.log(3))
GAMMA = 0.57721566490153286
CHI_QUIRAL = GAMMA / KAPPA_INFO
AIRY_RATIO = (4.087949444130970 / 2.338107410459767) ** (3/4)

def rho_brent_tsm(n, max_evals=50000):
    n_mpz = gmpy2.mpz(n)
    canal = n % 6
    # Inicialización Quiral (DeepSeek)
    if canal == 1: c = gmpy2.mpz(int(R_FUND * 10**6) % n_mpz)
    else: c = gmpy2.mpz(int(CHI_QUIRAL * 10**6) % n_mpz)
    if c == 0 or c == n_mpz-2: c = 1

    y = gmpy2.mpz(random.randint(2, int(n_mpz-2)))
    r = 1; m = 256; q = gmpy2.mpz(1); G = gmpy2.mpz(1); evals = 0

    # Bucle Seguro Original
    while G == 1 and evals < max_evals:
        x = y
        for _ in range(r): y = gmpy2.f_mod(y*y + c, n_mpz)
        k = 0
        while k < r and G == 1 and evals < max_evals:
            limite = min(m, r - k)
            for _ in range(limite):
                y = gmpy2.f_mod(y*y + c, n_mpz)
                q = gmpy2.f_mod(q * abs(x - y), n_mpz)
                evals += 1
            G = gmpy2.gcd(q, n_mpz)
            k += limite
        r *= 2

    if G == n_mpz:
        while True:
            y = gmpy2.f_mod(y*y + c, n_mpz)
            G = gmpy2.gcd(abs(x - y), n_mpz)
            if G > 1: break

    return int(G) if 1 < G < n_mpz else None

def calibrar_ecm_tsm(residuo):
    """
    Logística Corregida: Doctrina de Airy pura.
    Siempre empezamos en asedio ligero y escalamos geométricamente.
    """
    canal = residuo % 6
    friccion = 1.0 if canal == 1 else CHI_QUIRAL

    b1_base = 5000
    curvas_base = max(12, int(20 * friccion)) # La quiralidad modula la munición, no el calibre B1

    niveles = []
    max_oleadas = 12 if len(str(residuo)) > 80 else 6

    b1 = b1_base
    curvas = curvas_base
    for i in range(max_oleadas):
        niveles.append((int(b1), int(curvas)))
        # Escalamiento geométrico de Airy original
        b1 = b1 * AIRY_RATIO
        curvas = curvas * AIRY_RATIO

    return niveles

def ecm_worker_wrapper(n_bytes, sigma, b1, curvas):
    res = L_LIB.ecm_final_cpp(n_bytes, sigma, b1, curvas)
    return int(res.decode()) if res else None

# =========================================================================
# 3. ORQUESTADOR MAESTRO
# =========================================================================
class LeviatanFinal:
    def factorizar(self, N):
        if gmpy2.is_prime(gmpy2.mpz(N)): return [int(N)]
        factores = []

        # FASE 1: Vanguardia Z/6Z
        print(f"\n[💣] FASE 1: Vanguardia Z/6Z (Criba hasta 1e9)...")
        res = L_LIB.criba_divide_cpp(str(N).encode(), 1000000000).decode().split(',')
        residuo = int(res[-1]); factores.extend([int(p) for p in res[:-1] if p])
        if residuo == 1: return sorted(factores)
        if gmpy2.is_prime(gmpy2.mpz(residuo)): return sorted(factores + [residuo])

        # FASE 1.5: Infantería Rho
        print(f"[🧹] FASE 1.5: Infantería Rho (Calibración Quiral, 50k evals)...")
        start_rho = time.time()
        hit = rho_brent_tsm(residuo, 50000)
        if hit:
            print(f"   [⚡] Fractura Rho en {time.time()-start_rho:.2f}s: {hit}")
            return sorted(factores + self.factorizar(hit) + self.factorizar(residuo//hit))
        else:
            print(f"   [-] Área despejada ({time.time()-start_rho:.2f}s). El enemigo es pesado.")

        # FASE 2: Espada Z/6Z Completa
        canal_residuo = residuo % 6
        if canal_residuo in (1, 5):
            print(f"[⚔️] FASE 2: Espada de Alejandro (Canal {canal_residuo})...")
            start_espada = time.time()
            max_iters = int(1000000 * (len(str(residuo))/30) ** (-2))
            max_iters = max(10000, min(max_iters, 10000000))
            res = L_LIB.espada_cpp(str(residuo).encode(), canal_residuo, max_iters)
            if res:
                hit = int(res.decode())
                print(f"   [🗡️] Decapitación Geométrica en {time.time()-start_espada:.2f}s: {hit}")
                return sorted(factores + self.factorizar(hit) + self.factorizar(residuo//hit))
            else:
                print(f"   [-] Fuera de alcance geométrico ({time.time()-start_espada:.2f}s).")

        # FASE 3: Artillería Montgomery-Airy (Motor L1 Original)
        print(f"[🔥] FASE 3: Artillería Montgomery-Airy (Motor L1 + Parada Prematura)...")
        niveles = calibrar_ecm_tsm(residuo)
        n_bytes = str(residuo).encode()
        cores = multiprocessing.cpu_count()
        print(f"   [📐] Calibración TSM: {len(niveles)} oleadas geométricas detectadas.")

        for idx, (b1, curvas_total) in enumerate(niveles):
            curvas_por_core = max(1, curvas_total // cores)
            print(f"   └─ Oleada {idx+1}: B1={b1:,}, Curvas={curvas_por_core*cores}")
            sigma_base = 6.0 + idx * 100

            # EL PARCHE TÁCTICO: as_completed para atrapar la victoria al instante
            with ProcessPoolExecutor(max_workers=cores) as ex:
                tareas = [ex.submit(ecm_worker_wrapper, n_bytes, sigma_base + w, b1, curvas_por_core) for w in range(cores)]
                for f in as_completed(tareas):
                    res = f.result()
                    if res:
                        hit = res
                        print(f"   [⭐] ¡FRACTURA TERMODINÁMICA EN OLEADA {idx+1}!")
                        return sorted(factores + self.factorizar(hit) + self.factorizar(residuo//hit))

        return sorted(factores + [residuo])

# =========================================================================
# TEST DE VALIDACIÓN: EL CRIPTOGRAMA DE 104 DÍGITOS
# =========================================================================
if __name__ == "__main__":
    # Nuestro objetivo asimétrico original. Debería caer en menos de 60 segundos.
    objetivo = gmpy2.mpz("4567891718764363002247560149378699285181380785609370436049370436049368921185532719666532719666516183")
    print(f"\n🎯 OBJETIVO FINAL: {objetivo}\nMagnitud: {len(str(objetivo))} dígitos")
    start = time.time()
    l_final = LeviatanFinal()
    res = l_final.factorizar(objetivo)
    print("\n" + "="*60 + f"\n✅ BATALLA CONCLUIDA EN {time.time()-start:.2f}s\nFactores: {res}\n" + "="*60)

📦 Verificando blindaje y dependencias...
[🔨] Forjando Núcleo C++ Optimizado...
[✅] Núcleo operativo y blindado.

🎯 OBJETIVO FINAL: 4567891718764363002247560149378699285181380785609370436049370436049368921185532719666532719666516183
Magnitud: 100 dígitos

[💣] FASE 1: Vanguardia Z/6Z (Criba hasta 1e9)...
[🧹] FASE 1.5: Infantería Rho (Calibración Quiral, 50k evals)...
   [-] Área despejada (0.05s). El enemigo es pesado.
[⚔️] FASE 2: Espada de Alejandro (Canal 5)...
   [-] Fuera de alcance geométrico (0.00s).
[🔥] FASE 3: Artillería Montgomery-Airy (Motor L1 + Parada Prematura)...
   [📐] Calibración TSM: 12 oleadas geométricas detectadas.
   └─ Oleada 1: B1=5,000, Curvas=72
   └─ Oleada 2: B1=7,602, Curvas=110
   └─ Oleada 3: B1=11,559, Curvas=168
   └─ Oleada 4: B1=17,575, Curvas=256
   └─ Oleada 5: B1=26,723, Curvas=390
   └─ Oleada 6: B1=40,632, Curvas=592
   └─ Oleada 7: B1=61,780, Curvas=902
   └─ Oleada 8: B1=93,936, Curvas=1370
   [⭐] ¡FRACTURA TERMODINÁMICA EN OLEADA 8!

✅ BATALLA C

In [ ]:
import ctypes
import time
import os
import tracemalloc

# 1. Cargar la librería forjada en C++
LIB_PATH = os.path.abspath('./erato_z6z.so')
if not os.path.exists(LIB_PATH):
    raise FileNotFoundError("¡El núcleo erato_z6z.so ha desaparecido! Vuelve a forjarlo.")

motor_z6z = ctypes.CDLL(LIB_PATH)

# 2. Configurar el puente de datos nativo
# Recibe un uint64_t y devuelve un uint64_t
motor_z6z.criba_z6z_bitwise.argtypes = [ctypes.c_uint64]
motor_z6z.criba_z6z_bitwise.restype = ctypes.c_uint64

def asedio_mil_millones():
    LIMITE = 1_000_000_000 # 1 Billón (Escala Americana)

    print("="*60)
    print(f" 🚀 PRUEBA DE ESTRÉS: ERATÓSTENES Z/6Z BITWISE C++")
    print("="*60)
    print(f"Objetivo: Identificar todos los primos hasta {LIMITE:,}")
    print("[⚙️] Desatando el motor nativo de hardware...")

    # Iniciar telemetría
    tracemalloc.start()
    start = time.time()

    # Impacto principal llamando al código C++
    total_primos = motor_z6z.criba_z6z_bitwise(LIMITE)

    # Detener telemetría
    tiempo = time.time() - start
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    memoria_mb = peak_memory / (1024 * 1024)

    print("\n" + "-" * 60)
    print(f"✅ BARRIDO COMPLETADO")
    print(f"   ├─ Primos encontrados: {total_primos:,}")
    print(f"   ├─ Tiempo de asedio:   {tiempo:.4f} segundos")
    print(f"   └─ Memoria Pico (RAM): {memoria_mb:.2f} MB")
    print("-" * 60)

    # Verificación de Integridad Histórica
    # Sabemos matemáticamente que hay exactamente 50,847,534 primos hasta 1,000,000,000
    if total_primos == 50847534:
        print("🔍 INTEGRIDAD Z/6Z: PERFECTA (El número exacto de primos es correcto)")
    else:
        print(f"❌ ERROR: Discrepancia detectada. Se esperaban 50,847,534 y se obtuvieron {total_primos:,}")

# Ejecutar orden
if __name__ == "__main__":
    asedio_mil_millones()

In [ ]:
import math

def analizador_tactico_z6z(N):
    """
    Escáner de Reconocimiento Termodinámico basado en el Sustrato Z/6Z.
    Analiza el criptograma antes del asedio.
    """
    print("="*60)
    print(" 🔍 RECONOCIMIENTO TÁCTICO Z/6Z ")
    print("="*60)

    digitos = len(str(N))
    bits = N.bit_length()
    canal_N = N % 6

    print(f"[!] Objetivo: {N}")
    print(f"[!] Magnitud: {digitos} dígitos | {bits} bits")
    print(f"[!] Canal del Criptograma: {canal_N} (mod 6)\n")

    # 1. DEDUCCIÓN TOPOLÓGICA DE FACTORES
    print("--- 1. ANÁLISIS TOPOLÓGICO DE FACTORES ---")
    if canal_N == 5:
        print("✔️  DIAGNÓSTICO: Asimetría Quiral Detectada.")
        print("   └─ Obligatorio: Un factor pertenece a C1 (≡ 1 mod 6) y el otro a C5 (≡ 5 mod 6).")
        print("   └─ Fricción Termodinámica: Mínima. La Espada de Alejandro (Fermat) operará a máxima eficiencia (salto x3).")
        friccion = 1.0
    elif canal_N == 1:
        print("✔️  DIAGNÓSTICO: Simetría Quiral Detectada.")
        print("   └─ Obligatorio: Ambos factores pertenecen a C1, o ambos pertenecen a C5.")
        print("   └─ Fricción Termodinámica: Máxima. Se requiere orquestación ECM profunda.")
        friccion = 3.65946 # Tu constante CHI_QUIRAL
    else:
        print("❌ DIAGNÓSTICO: Criptograma Débil.")
        print("   └─ El número contiene factores triviales (2 o 3). No es un RSA válido.")
        return

    # 2. DIMENSIÓN FRACTAL Y BITS EFECTIVOS
    print("\n--- 2. FÍSICA DEL VACÍO ARITMÉTICO ---")
    D2 = 0.24338
    bits_efectivos = bits * D2
    print(f"✔️  Dimensión Fractal del Soporte (D2): {D2}")
    print(f"✔️  Espacio de Búsqueda Lineal: 2^{bits} operaciones.")
    print(f"✔️  Espacio Efectivo Z/6Z: 2^{bits_efectivos:.2f} operaciones.")
    print(f"   └─ El sustrato ha comprimido la entropía en un {(1 - D2)*100:.2f}%.")

    # 3. RECOMENDACIÓN DE ASEDIO
    print("\n--- 3. DOCTRINA DE ASEDIO RECOMENDADA ---")
    if bits < 150: # Hasta ~45 dígitos
        print("🚀 ARMA: Leviatán Fase 2 (Espada de Alejandro + ECM Ligero)")
        print("   └─ Tiempo estimado: Sub-minuto.")
    elif bits < 300: # Hasta ~90 dígitos
        print("🔥 ARMA: Leviatán Fase 3 (ECM Montgomery-Airy con Parada Prematura)")
        print(f"   └─ Calibración: {int(bits_efectivos * 0.6 * friccion)} curvas iniciales requeridas.")
        print("   └─ Tiempo estimado: Minutos / Horas tempranas.")
    else:
        print("💀 ARMA: MPQS / GNFS (Criba Cuadrática/Cuerpo de Números)")
        print("   └─ Límite térmico alcanzado para ECM de un solo nodo. Transición a álgebra matricial requerida.")
    print("="*60)

# Prueba del escáner con un número de 45 dígitos
objetivo_test = 1522605027922533360535618378132637429718068114961380688657908494580122963258952897654000350692006139
#objetivo_test = 123456789012345678901234567890123456789012343
analizador_tactico_z6z(objetivo_test)